# Sepsis dataset exploratory analysis

This notebook loads the processed six-hour windows and exports presentation-ready figures to `outputs/eda/`. The missingness chart reads the original PSV files so it represents data **before** imputation.

In [ ]:
import os
from pathlib import Path

# Support launching Jupyter from either the project root or notebooks/.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if (CURRENT_DIR.parent / "data").exists() else CURRENT_DIR
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs" / ".matplotlib"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "data.npz"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "eda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with np.load(DATA_PATH, allow_pickle=False) as data:
    X = data["X"]
    y = data["y"]
    feature_names = data["feature_names"].astype(str).tolist()

print(f"Loaded X={X.shape}, y={y.shape}")
print(f"Writing figures to: {OUTPUT_DIR}")

## 1. Class balance

In [ ]:
class_counts = pd.Series(y).value_counts().reindex([0, 1], fill_value=0)
plot_data = pd.DataFrame({"Label": ["Non-sepsis", "Sepsis"], "Windows": class_counts.values})

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(data=plot_data, x="Label", y="Windows", hue="Label", palette=["#4C78A8", "#E45756"], legend=False, ax=ax)
for bar, count in zip(ax.patches, plot_data["Windows"]):
    ax.annotate(f"{count:,} ({count / len(y):.1%})", (bar.get_x() + bar.get_width() / 2, bar.get_height()), ha="center", va="bottom", fontsize=12)
ax.set_title("Window-level sepsis class balance")
ax.set_ylabel("Number of 6-hour windows")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "class_balance.png", dpi=200, bbox_inches="tight")
plt.show()

## 2. Key vital-sign distributions

Each observation below is the final ICU hour of a six-hour window, matching the window label. Values are z-score normalized by the preprocessing script.

In [ ]:
key_vitals = [vital for vital in ["HR", "Temp", "Resp", "MAP"] if vital in feature_names]
if not key_vitals:
    raise ValueError(f"None of HR, Temp, Resp, or MAP were found. Available features: {feature_names}")

last_hour = X[:, -1, :]
vital_frame = pd.DataFrame({vital: last_hour[:, feature_names.index(vital)] for vital in key_vitals})
vital_frame["Label"] = np.where(y == 1, "Sepsis", "Non-sepsis")
long_vitals = vital_frame.melt(id_vars="Label", var_name="Vital sign", value_name="Z-score")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for axis, vital in zip(axes.flat, key_vitals):
    sns.histplot(
        data=long_vitals[long_vitals["Vital sign"] == vital],
        x="Z-score", hue="Label", stat="density", common_norm=False,
        bins=35, element="step", fill=True, alpha=0.3, palette=["#4C78A8", "#E45756"], ax=axis,
    )
    axis.set_title(vital)
for axis in axes.flat[len(key_vitals):]:
    axis.remove()
fig.suptitle("Key vital-sign distributions at each window's last hour", y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "vital_distributions_by_label.png", dpi=200, bbox_inches="tight")
plt.show()

## 3. Raw missing-data pattern

To keep the figure readable, this samples the first 20 patients and at most 240 ICU-hour rows. Yellow cells indicate missing raw measurements before any forward-fill, back-fill, or zero fill.

In [ ]:
raw_files = sorted(RAW_DIR.glob("*.psv"))
if not raw_files:
    raise FileNotFoundError(f"No raw .psv files found in {RAW_DIR}")

raw_frames = [pd.read_csv(path, sep="|") for path in raw_files[:20]]
raw_sample = pd.concat(raw_frames, ignore_index=True).head(240)
available_features = [feature for feature in feature_names if feature in raw_sample.columns]
missing = raw_sample[available_features].isna().T

fig, ax = plt.subplots(figsize=(18, max(8, len(available_features) * 0.28)))
sns.heatmap(missing, cmap=["#F7F7F7", "#F2C14E"], cbar_kws={"label": "Missing raw value"}, yticklabels=True, xticklabels=False, ax=ax)
ax.set_title("Raw feature missingness before imputation (sample of ICU hours)")
ax.set_xlabel("ICU-hour observations")
ax.set_ylabel("Feature")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "raw_missingness_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()